In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import layers


from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras    
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

2024-03-22 08:11:23.630096: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-22 08:11:24.730617: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-03-22 08:11:27.120332: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:27.342584: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:27.342643: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:27.497454: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:27.497518: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 12367864894117250756
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 5832462868004604941
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [3]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [2]:
import os
os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")

0

In [5]:
import pandas as pd

In [4]:
reader = pd.read_csv("/root/Data/hangul/dataset/test.csv", encoding='utf-8', header=None)
idx = 1000
print(reader.iloc[idx, :][0])
ja = label2ja[reader.iloc[idx, :][1]]
mo = label2mo[reader.iloc[idx, :][2]]
ba = label2ba[reader.iloc[idx, :][3]]

char = unicode.join_jamos_char(ja, mo ,ba)
print(char)


NameError: name 'pd' is not defined

In [4]:
def get_dataset_fromCsv():
    cnt = 0
    print("Helelo?")
    
    csvFile = open("/root/Data/hangul/dataset/tranDataset.csv", 'r', encoding='utf-8')
    os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        yield img, (label1, label2, label3)

In [5]:
def get_valid_dataset_fromCsv():
    cnt = 0
    print("Helelo?")
    
    csvFile = open("/root/Data/hangul/dataset/validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)

        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        yield img, (label1, label2, label3)

In [6]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

# print(dataset)
# iterator = iter(dataset)
# print(next(iterator))

2024-03-21 08:37:58.492088: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-21 08:37:58.492164: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-21 08:37:58.492192: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-21 08:37:58.492552: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-21 08:37:58.492587: I external/local_xla/xla/stream_executor

In [4]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

#tf.keras.layers.Conv2D

# l = layers.Conv2D(filters=48, kernel_size=(3,3), activation = 'relu', strides = (1,1))(posts_input)
# l = layers.MaxPooling2D(pool_size=(2,2), strides=2, padding="same")(l)
 
# l = layers.Conv2D(filters=64, kernel_size=(3,3), activation = 'relu', strides = (1,1))(l)
# l = layers.MaxPooling2D(padding="same")(l)

l = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(posts_input)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)


x = layers.Flatten()(l)
#x = layers.Flatten()(posts_input)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

2024-03-22 08:11:37.456083: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:37.456178: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:37.456209: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:37.456591: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-22 08:11:37.456629: I external/local_xla/xla/stream_executor

In [1]:
# folder_directory = "체크포인트를 저장할 폴더"
# checkPoint_path = folder_directory+"/model_{epoch}.ckpt" # 저장할 당시 epoch가 파일이름이 된다.

# # 2. 콜백 변수를 생성
# my_period = 몇번의 학습마다 저장할 것인가?
# cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkPoint_path,
# 					save_weights_only=True, verbose=1, period=my_period) 
    
# # 3. fit의 파라미터에 callbacks생성
# model.fit(self.x_train, self.y_train, batch_size=30, epochs=100, validation_split=0.2, 
#     	callbacks=[cp_callback],  verbose=1) 

save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwrite.weights.h5"


#dataset = dataset.repeat().batch(16)
#validDtaset = validDtaset.repeat().batch(16)
#dataset = dataset.batch(16)
#validDtaset = validDtaset.batch(16)

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')


#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
#model.fit(validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= dataset)
#model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

NameError: name 'keras' is not defined

In [8]:
#dataset = tf.data.Dataset.from_tensor_slices(({'input_x': data_a, 'input_y': data_b}, labels)).batch(2).repeat()
#model.fit(get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv"), epochs = 100, batch_size = 16)
#dataset = dataset.shuffle(150).batch(8)
model.fit(dataset, validation_data= validDtaset, epochs = 20, batch_size = 16)
#model.fit_generator(dataset, epochs = 100)


Epoch 1/20
Helelo?


I0000 00:00:1710977903.710212   10678 service.cc:145] XLA service 0x7f5780003900 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1710977903.710250   10678 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-03-21 08:38:23.766227: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-21 08:38:24.025133: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


     25/Unknown 6s 7ms/step - DenseCho2_accuracy: 0.1181 - DenseJong2_accuracy: 0.0724 - DenseJung2_accuracy: 0.0128 - loss: 57.9877       

I0000 00:00:1710977906.148273   10678 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 276884/Unknown 1725s 6ms/step - DenseCho2_accuracy: 0.6137 - DenseJong2_accuracy: 0.7550 - DenseJung2_accuracy: 0.5388 - loss: 3.4792

...

2024-03-21 18:01:11.275205: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-21 18:01:11.275249: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-03-21 18:01:11.275261: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 12607808536000978135
2024-03-21 18:01:11.275266: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1482904278368540353
2024-03-21 18:01:11.275272: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14790398889878133132
2024-03-21 18:01:11.275295: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9516691308485560712


276880/276886 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - DenseCho2_accuracy: 0.9879 - DenseJong2_accuracy: 0.9866 - DenseJung2_accuracy: 0.9745 - loss: 0.2848Helelo?


2024-03-21 18:29:50.873492: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-21 18:29:50.873563: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-21 18:29:50.873599: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9516691308485560712


276886/276886 ━━━━━━━━━━━━━━━━━━━━ 1822s 7ms/step - DenseCho2_accuracy: 0.9879 - DenseJong2_accuracy: 0.9866 - DenseJung2_accuracy: 0.9745 - loss: 0.2848 - val_DenseCho2_accuracy: 0.9054 - val_DenseJong2_accuracy: 0.8891 - val_DenseJung2_accuracy: 0.8698 - val_loss: 3.1862


2024-03-21 18:31:33.655625: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-21 18:31:33.655660: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-03-21 18:31:33.655671: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 12607808536000978135
2024-03-21 18:31:33.655675: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1482904278368540353
2024-03-21 18:31:33.655680: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14790398889878133132
2024-03-21 18:31:33.655701: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9516691308485560712


In [11]:
model.save_weights('han_master5.weights.h5')

In [58]:
model.load_weights("han_master5.weights.h5")

img = tf.io.read_file("/root/Data/hangul_handwrite_val/image/test/35.jpg")
#img = tf.io.read_file("/root/Data/hangul_handwrite_val/image//31.jpg")
#img = tf.io.read_file("E:\\unzipData\\Training\\image_Training_handwrite\\1.letter\\040\\04030003097")
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.convert_image_dtype(img, tf.float32)
img = tf.image.resize(img, (64, 64))

print(img.shape)

img = np.array(img)

img = np.expand_dims(img, axis=0)

#print(img.shape)

ch, ju, jo = model.predict(img)

ch = ch.argmax()
ju = ju.argmax()
jo = jo.argmax()

ja = label2ja[ch]
mo = label2mo[ju]
ba = label2ba[jo]

char = unicode.join_jamos_char(ja, mo ,ba)
print(char)

(64, 64, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
솨
